### NanoGPT Implementation
- Implementation of Karpathy's lecture on building GPT from scratch: https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLAqhIrjkxbuWI23v9cThsA9GvCAUhRvKZ&index=10

- Google collab from lecture: https://colab.research.google.com/drive/1JMLa53HDuA-i7ZBmqV7ZnA3c_fvtXnx-?usp=sharing#scrollTo=O6medjfRsLD9


### Get input data

In [ ]:
# import urllib.request

# url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
# urllib.request.urlretrieve(url, "input.txt")

### Build data processing and model in pieces

In [ ]:
# read it in to inspect it
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

In [ ]:
print(text[:1000])

In [ ]:
# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

print("".join(chars))

In [ ]:
# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}

# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

print(encode("my name is zeal"))
print(decode(encode("my name is zeal")))

In [ ]:
# train and test splits
import torch

data = torch.tensor(encode(text), dtype=torch.long)

# 90% training and 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Data size: Train = {len(train_data)} | Val = {len(val_data)}")

In [ ]:
# extract one block of training examples
# a single block contains block_size number of examples

block_size = 8  # same as context_length

x = train_data[:block_size]  # single block
y = train_data[1 : block_size + 1]


print("Single block of example packs block_size number of examples")
for i in range(block_size):
    context = x[: i + 1]
    target = y[i]
    print(f"Example {i} --> context = {context} and target = {target} ")


In [ ]:
# data loader
block_size = 8
batch_size = 4


# output dimension: (B,T) where B (batch dim) is batch_size, T (time dim) is block_size
def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    return x, y


xb, yb = get_batch(split="train")

print(f"Inputs: {x.shape} \n {x}")
print(f"Targets: {y.shape} \n {y}")

# -------------------------------------------------------------------------------------------------
# visualize every example packed in these four batches for our DECODER transformer block
eg = 0
for batch in range(batch_size):
    for time in range(block_size):
        context = xb[batch, 0 : time + 1]
        target = yb[batch, time]
        print(f"Example {eg} --> context = {context} and target = {target}")
        eg += 1


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1266)

In [ ]:
# start with bigram model. Bigram model uses a (vocab_size, vocab_size) embedding matrix
# passing "x" of size (3,4) to this matrix --> bigram_embedding_matrix(x) --> it will return a (3,4,vocab_size) output
# where, each element of x is now embedded into vocab_size dimensions


class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, x, targets=None):
        # x is of size (B,T), where each element is a character index from the vocabulary
        # target is of size (B,T)

        logits = self.token_embedding_table(x)  # (B,T,vocab_size) after passing through the embedding table

        # loss
        if targets == None:
            loss = None
        else:
            # pytorch requires shape (B,C,T) instead of (B,T,C), which is confusing, so we combine the first two dims.
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, x, max_new_tokens):
        # generate one token at a time
        for i in range(max_new_tokens):
            # forward pass through the model
            logits, _ = self.forward(x)  # logits is (B,T,vocab_size)
            # focus only on the last timestep; no dependence on past here
            logits = logits[:, -1, :]  # (B,vocab_size)
            # convert logits to probabilities
            probs = F.softmax(logits, dim=-1)
            # sample next token from probability distribution
            next_token = torch.multinomial(probs, num_samples=1)  # (B,1)
            # append next_token to the original context
            x = torch.cat((x, next_token), dim=1)  # (B,T+1)
        return x


# call the model
model = BigramModel()
logits, loss = model(x=xb, targets=yb)
print(f"Loss = {loss}")

model_output = model.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)
print(decode(model_output[0].tolist()))

In [ ]:
# train the bigram model

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for steps in range(1000):
    xb, yb = get_batch("train")

    _, loss = model(x=xb, targets=yb)
    # set all gradients None
    optimizer.zero_grad(set_to_none=True)
    # run backward pass
    loss.backward()
    # make gradient update for each parameter
    optimizer.step()

print(loss.item())


In [ ]:
model_output = model.generate(x=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=1000)
print(decode(model_output[0].tolist()))

### Adding positional encoding and single self-attention head with ffn

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1266)

# ---------------------------Get data ready---------------------------
# read input data
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}
# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

# train-val split: 90% training and 10% validation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# ---------------------------hyperparameters---------------------------
# hyperparameters
block_size = 8
batch_size = 4
lr = 1e-3
n_embd = 64
head_size = 8
max_iters = 10000
# Check for CUDA (NVIDIA), then MPS (Apple Silicon), otherwise default to CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")
# ---------------------------------------------------------------------


# data loader
# output dimension: (B,T) where B (batch dim) is batch_size, T (time dim) is block_size
def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    x, y = x.to(device), y.to(device)
    return x, y


# add single self attention head to bigram model
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril is pre-computed for a head as a fixed-size lower triangular matrix of size (block_size, block_size)
        self.register_buffer(name="lower_triag_matrix", tensor=torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape  # here, C is n_embd --> x is of shape (B,T,n_embd)
        q = self.query(x)  # (B,T,head_size)
        k = self.key(x)  # (B,T,head_size)
        v = self.value(x)  # (B,T,head_size)

        # calculate scaled weight matrix by wei by head_size
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)  # (B,T,head_size) @ (B,head_size,T) --> (B,T,T)

        # masking the weight matrix
        # during training or inference, your input batch might have a sequence length T
        # that is shorter than block_size (for example, T=16 while block_size = 1024)
        # hence we use self.tril[:T,:T]==0 and not self.tril==0
        wei = wei.masked_fill(mask=self.lower_triag_matrix[:T, :T] == 0, value=float("-inf"))  # (B,T,T)
        # softmax
        wei = F.softmax(wei, dim=-1)  # (B,T,T)

        # compute output
        out = wei @ v  # (B,T,head_size)
        return out


# main model class
class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # self attention head
        self.sa_head = Head(head_size=head_size)
        self.lm_head = nn.Linear(head_size, vocab_size)  # projection layer from head_size -> vocab_size

    def forward(self, x, targets=None):
        B, T = x.shape
        # information and positional encoding of input are computed and added
        tok_emb = self.token_embedding_table(x)  # (B,T,n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=x.device))  # (T,n_embd)
        x = tok_emb + pos_emb  # (B,T,n_embd)

        # single self-attention head
        x = self.sa_head(x)  # (B,T,head_size)
        # FFN
        # why FFN here: if we pass head's output directly to cross-entropy it will error out because
        # head's output is in range 0 to head_size, where as target is in range 0 to vocab_size.
        # what does it mean: (B,T,vocab_size) --> stores logit values for all vocabulary tokens at each position.
        logits = self.lm_head(x)  # (B,T,vocab_size)

        # loss
        if targets == None:
            loss = None
        else:
            # pytorch requires shape (B,C,T) instead of (B,T,C), which is confusing, so we combine the first two dims.
            # Remember: logits (B,T,vocab_size) matrix stores logit values for all vocabulary tokens at each position.
            # Cross-entropy converts logits to probabilities for each vocabulary token at every position.
            # It then selects the probability of the target token to compute the loss.
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, input, max_new_tokens):
        """
        Generates new tokens using past context.

        This function predicts next tokens one by one up to max_new_tokens.
        It crops the input context to the maximum block size before each step.

        Parameters:
            input (torch.Tensor): Tensor of shape (B, T) with current token indices.
            max_new_tokens (int): The number of new tokens to generate.

        Returns:
            torch.Tensor: Tensor of shape (B, T + max_new_tokens) with full sequence.
        """
        for _ in range(max_new_tokens):
            # limit context to latest block_size tokens
            input_clipped_to_context_window = input[:, -block_size:]

            # forward pass this temporary slice through the model
            logits, _ = self.forward(x=input_clipped_to_context_window, targets=None)  # logits is (B,T,vocab_size)

            # focus only on the last timestep; no dependence on past here
            # Reason: through self-attention trick, the last position/timestep already takes into account
            # all the past positions for every batch.
            logits = logits[:, -1, :]  # (B,vocab_size)

            # convert logits to probabilities
            # for every vocab token for each batch, we get one probability value
            probs = F.softmax(logits, dim=-1)  # (B,vocab_size)

            # sample next token from probability distribution for every batch
            next_token = torch.multinomial(probs, num_samples=1)  # (B,1)

            # append next_token to the original context
            input = torch.cat((input, next_token), dim=1)  # (B,T+1)
        return input


# -------------------------------single forward pass for testing-------------------------------
# model = BigramModel()
# m = model.to(device)
# # number of parameters in the model
# print(sum(p.numel() for p in m.parameters()) / 1e6, "M parameters")

## single forward pass
# xb, yb = get_batch(split="train")
# logits, loss = m(x=xb, targets=yb)
# print(f"Loss = {loss}")

# -------------------------------train the model e2e-------------------------------

# call the model
model = BigramModel()
m = model.to(device)

# number of parameters in the model
print(sum(p.numel() for p in m.parameters()) / 1e6, "M parameters")

optimizer = torch.optim.AdamW(m.parameters(), lr=lr)

for step in range(max_iters):
    xb, yb = get_batch("train")

    _, loss = m(x=xb, targets=yb)
    # set all gradients None
    optimizer.zero_grad(set_to_none=True)
    # run backward pass
    loss.backward()
    # make gradient update for each parameter
    optimizer.step()

    # print loss every few iterations
    if step % 1000 == 0:
        print(f"Iter: {step} | Loss: {loss.item()}")

print(f"Final Loss: {loss.item()}")

# -------------------------------generate using the model -------------------------------
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(input=context, max_new_tokens=2000)[0].tolist()))


### Adding multi-head attention, FFN and wrap them in blocks

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1266)

# ---------------------------Get data ready---------------------------
# read input data
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}
# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

# train-val split: 90% training and 10% validation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# ---------------------------hyperparameters---------------------------
# hyperparameters
block_size = 8
batch_size = 4
lr = 1e-3
n_embd = 64
# head_size = 8 # not using this since we are using head size = n_embd in the GPT paper
n_head = 4
n_layers = 4  # number of transformer blocks
max_iters = 10000

# Check for CUDA (NVIDIA), then MPS (Apple Silicon), otherwise default to CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")
# ---------------------------------------------------------------------


# data loader
# output dimension: (B,T) where B (batch dim) is batch_size, T (time dim) is block_size
def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    x, y = x.to(device), y.to(device)
    return x, y


# add single self attention head to bigram model
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril is pre-computed for a head as a fixed-size lower triangular matrix of size (block_size, block_size)
        self.register_buffer(name="lower_triag_matrix", tensor=torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape  # here, C is n_embd --> x is of shape (B,T,n_embd)
        q = self.query(x)  # (B,T,head_size)
        k = self.key(x)  # (B,T,head_size)
        v = self.value(x)  # (B,T,head_size)

        # calculate scaled weight matrix by wei by head_size
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)  # (B,T,head_size) @ (B,head_size,T) --> (B,T,T)

        # masking the weight matrix
        # during training or inference, your input batch might have a sequence length T
        # that is shorter than block_size (for example, T=16 while block_size = 1024)
        # hence we use self.tril[:T,:T]==0 and not self.tril==0
        wei = wei.masked_fill(mask=self.lower_triag_matrix[:T, :T] == 0, value=float("-inf"))  # (B,T,T)
        # softmax
        wei = F.softmax(wei, dim=-1)  # (B,T,T)

        # compute output
        out = wei @ v  # (B,T,head_size)
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, individual_head_size, block_size):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(n_embd=n_embd, head_size=individual_head_size, block_size=block_size) for _ in range(n_head)]
        )

    def forward(self, x):
        # x will be of shape (B,T,n_embd)
        # every single head(x) will return output of shape (B,T,individual_head_size)

        # stacking every head's output into a multi-head attn output will
        # return (B,T,inidividual_head_size*num_heads) i.e. (B,T,n_embd) back again
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return out


class FeedForwardNetwork(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.ffn = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        # x will come from multi-attn head with shape (B,T,head_size)
        out = self.ffn(x)  # (B,T,n_embd)
        return out


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        # multi-head attention (communication mode)
        individual_head_size = n_embd // n_head
        self.ma_head = MultiHeadAttention(
            n_embd=n_embd, n_head=n_head, individual_head_size=individual_head_size, block_size=block_size
        )
        # ffn (computation mode)
        self.ffn = FeedForwardNetwork(n_embd=n_embd)

    def forward(self, x):
        # x --> (B,T,n_embd)
        x = self.ma_head(x)  # (B,T,n_embd)
        x = self.ffn(x)  # (B,T,n_embd)
        return x


# main model class
class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # transformer block
        # note that "*" unpacks list of blocks into individual, comma-separated positional arguments as required by nn.Sequential
        # nn.Sequential(*[block1, block2, block3]) is equivalent to nn.Sequential(block1, block2, block3)
        self.blocks = nn.Sequential(
            *[Block(n_embd=n_embd, n_head=n_head, block_size=block_size) for _ in range(n_layers)]
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)  # final projection layer from head_size -> vocab_size

    def forward(self, x, targets=None):
        B, T = x.shape
        # information and positional encoding of input are computed and added
        tok_emb = self.token_embedding_table(x)  # (B,T,n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=x.device))  # (T,n_embd)
        x = tok_emb + pos_emb  # (B,T,n_embd)

        # block (multi-attn head + FFN)
        x = self.blocks(x)  # (B,T,n_embd)

        # Final FFN
        # why FFN at the end: if we pass head's output directly to cross-entropy it will error out because
        # head's output is in range 0 to head_size, where as target is in range 0 to vocab_size.
        # what does it mean: (B,T,vocab_size) --> stores logit values for all vocabulary tokens at each position.
        logits = self.lm_head(x)  # (B,T,vocab_size)

        # loss
        if targets == None:
            loss = None
        else:
            # pytorch requires shape (B,C,T) instead of (B,T,C), which is confusing, so we combine the first two dims.
            # Remember: logits (B,T,vocab_size) matrix stores logit values for all vocabulary tokens at each position.
            # Cross-entropy converts logits to probabilities for each vocabulary token at every position.
            # It then selects the probability of the target token to compute the loss.
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, input, max_new_tokens):
        """
        Generates new tokens using past context.

        This function predicts next tokens one by one up to max_new_tokens.
        It crops the input context to the maximum block size before each step.

        Parameters:
            input (torch.Tensor): Tensor of shape (B, T) with current token indices.
            max_new_tokens (int): The number of new tokens to generate.

        Returns:
            torch.Tensor: Tensor of shape (B, T + max_new_tokens) with full sequence.
        """
        for _ in range(max_new_tokens):
            # limit context to latest block_size tokens
            input_clipped_to_context_window = input[:, -block_size:]

            # forward pass this temporary slice through the model
            logits, _ = self.forward(x=input_clipped_to_context_window, targets=None)  # logits is (B,T,vocab_size)

            # focus only on the last timestep; no dependence on past here
            # Reason: through self-attention trick, the last position/timestep already takes into account
            # all the past positions for every batch.
            logits = logits[:, -1, :]  # (B,vocab_size)

            # convert logits to probabilities
            # for every vocab token for each batch, we get one probability value
            probs = F.softmax(logits, dim=-1)  # (B,vocab_size)

            # sample next token from probability distribution for every batch
            next_token = torch.multinomial(probs, num_samples=1)  # (B,1)

            # append next_token to the original context
            input = torch.cat((input, next_token), dim=1)  # (B,T+1)
        return input


# -------------------------------single forward pass for testing-------------------------------
# model = BigramModel()
# m = model.to(device)
# # number of parameters in the model
# print(sum(p.numel() for p in m.parameters()) / 1e6, "M parameters")

## single forward pass
# xb, yb = get_batch(split="train")
# logits, loss = m(x=xb, targets=yb)
# print(f"Loss = {loss}")

# -------------------------------train the model e2e-------------------------------

# call the model
model = BigramModel()
m = model.to(device)

# number of parameters in the model
print(sum(p.numel() for p in m.parameters()) / 1e6, "M parameters")

optimizer = torch.optim.AdamW(m.parameters(), lr=lr)

for step in range(max_iters):
    xb, yb = get_batch("train")

    _, loss = m(x=xb, targets=yb)
    # set all gradients None
    optimizer.zero_grad(set_to_none=True)
    # run backward pass
    loss.backward()
    # make gradient update for each parameter
    optimizer.step()

    # print loss every few iterations
    if step % 1000 == 0:
        print(f"Iter: {step} | Loss: {loss.item()}")

print(f"Final Loss: {loss.item()}")

# -------------------------------generate using the model -------------------------------
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(input=context, max_new_tokens=2000)[0].tolist()))


### Adding skip connections, layernorm, dropout

In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(1266)

# ---------------------------Get data ready---------------------------
# read input data
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# unique characters found in text
chars = sorted(set(text))
vocab_size = len(chars)

# map chars to integers and vice versa
stoi = {st: i for i, st in enumerate(chars)}
itos = {i: st for i, st in enumerate(chars)}
# functions to encode and decode a given sequence using stoi and itos mapping
encode = lambda x: [stoi[c] for c in x]
decode = lambda x: "".join([itos[i] for i in x])

# train-val split: 90% training and 10% validation
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# ---------------------------hyperparameters---------------------------
# hyperparameters
block_size = 32
batch_size = 16
lr = 1e-3
n_embd = 64
# head_size = 8 # not using this since we are using head size = n_embd in the GPT paper
n_head = 4
n_layers = 4  # number of transformer blocks
dropout = 0.1
max_iters = 5000
eval_iters = 200
eval_interval = 100

# Check for CUDA (NVIDIA), then MPS (Apple Silicon), otherwise default to CPU
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")
# ---------------------------------------------------------------------


# data loader
# output dimension: (B,T) where B (batch dim) is batch_size, T (time dim) is block_size
def get_batch(split):
    data = train_data if split == "train" else val_data
    # randomly pick one index per batch.
    batch_idx = torch.randint(low=0, high=len(data) - block_size, size=(batch_size,))
    # extract each batch of examples that start at their respective idx and end at idx+block_size
    x = torch.stack([data[ix : ix + block_size] for ix in batch_idx])
    y = torch.stack([data[ix + 1 : ix + block_size + 1] for ix in batch_idx])
    x, y = x.to(device), y.to(device)
    return x, y


# below layernorm code is for reference; it is similar to torch's nn.LayerNorm()
# we will use torch's layernorm for all purposes
class LayerNorm1d(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))  # dim is n_embd so --> shape (n_embd)
        self.beta = nn.Parameter(torch.zeros(dim))  # shape (n_embd)

    def forward(self, x):
        # x will be (B,T,n_embd)
        xmean = x.mean(dim=-1, keepdim=True)  # (B,T,1)
        xvar = x.var(dim=-1, keepdim=True, unbiased=True)  # (B,T,1)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)  # (B,T,n_embd)
        out = self.gamma * xhat + self.beta  # (n_embd) * (B,T,n_embd) + (n_embd)
        return out


# add single self attention head to bigram model
class Head(nn.Module):
    def __init__(self, n_embd, head_size, block_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # tril is pre-computed for a head as a fixed-size lower triangular matrix of size (block_size, block_size)
        self.register_buffer(name="lower_triag_matrix", tensor=torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # here, C is n_embd --> x is of shape (B,T,n_embd)
        q = self.query(x)  # (B,T,head_size)
        k = self.key(x)  # (B,T,head_size)
        v = self.value(x)  # (B,T,head_size)

        # calculate scaled weight matrix by wei by head_size
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)  # (B,T,head_size) @ (B,head_size,T) --> (B,T,T)

        # masking the weight matrix
        # during training or inference, your input batch might have a sequence length T
        # that is shorter than block_size (for example, T=16 while block_size = 1024)
        # hence we use self.tril[:T,:T]==0 and not self.tril==0
        wei = wei.masked_fill(mask=self.lower_triag_matrix[:T, :T] == 0, value=float("-inf"))  # (B,T,T)
        # softmax
        wei = F.softmax(wei, dim=-1)  # (B,T,T)

        # add dropout
        wei = self.dropout(wei)

        # compute output
        out = wei @ v  # (B,T,head_size)
        return out


class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, individual_head_size, block_size):
        super().__init__()
        self.heads = nn.ModuleList(
            [Head(n_embd=n_embd, head_size=individual_head_size, block_size=block_size) for _ in range(n_head)]
        )
        self.skip_proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x will be of shape (B,T,n_embd)
        # every single head(x) will return output of shape (B,T,individual_head_size)

        # stacking every head's output into a multi-head attn output will
        # return (B,T,inidividual_head_size*num_heads) i.e. (B,T,n_embd) back again
        out = torch.cat([head(x) for head in self.heads], dim=-1)

        # pass multi-attn head's output thru skip connection's projection layer
        out = self.skip_proj(x)  # (B,T,n_embd)
        out = self.dropout(out)
        return out


class FeedForwardNetwork(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        # standard multi-attn linear layer --> relu --> projection layer for skip connection
        self.ffn = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.ReLU(), nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout)
        )

    def forward(self, x):
        # x will come from multi-attn head with shape (B,T,head_size)
        out = self.ffn(x)  # (B,T,n_embd)
        return out


class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        # multi-head attention (communication mode)
        individual_head_size = n_embd // n_head
        self.ma_head = MultiHeadAttention(
            n_embd=n_embd, n_head=n_head, individual_head_size=individual_head_size, block_size=block_size
        )
        # ffn (computation mode)
        self.ffn = FeedForwardNetwork(n_embd=n_embd)
        # layernorm
        self.ln1 = nn.LayerNorm(n_embd)  # LayerNorm1d(dim=n_embd) for our implementation
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # x --> (B,T,n_embd)
        x = x + self.ma_head(self.ln1(x))  # (B,T,n_embd)
        x = x + self.ffn(self.ln2(x))  # (B,T,n_embd)
        return x


# main model class
class BigramModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # transformer block
        # note that "*" unpacks list of blocks into individual, comma-separated positional arguments as required by nn.Sequential
        # nn.Sequential(*[block1, block2, block3]) is equivalent to nn.Sequential(block1, block2, block3)
        self.blocks = nn.Sequential(
            *[Block(n_embd=n_embd, n_head=n_head, block_size=block_size) for _ in range(n_layers)]
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)  # final projection layer

    def forward(self, x, targets=None):
        B, T = x.shape
        # information and positional encoding of input are computed and added
        tok_emb = self.token_embedding_table(x)  # (B,T,n_embd)
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=x.device)
        )  # (T,n_embd) i.e. each position 0 to T-1 will be embedded in n_embd dimensions
        x = tok_emb + pos_emb  # (B,T,n_embd)

        # multiple blocks (multi-attn head + FFN) in series.
        x = self.blocks(x)  # (B,T,n_embd)

        # Final FFN
        # why FFN at the end: if we pass head's output directly to cross-entropy it will error out because
        # head's output is in range 0 to head_size, where as target is in range 0 to vocab_size.
        # what does it mean: (B,T,vocab_size) --> stores logit values for all vocabulary tokens at each position.
        logits = self.lm_head(x)  # (B,T,vocab_size)

        # loss
        if targets == None:
            loss = None
        else:
            # pytorch requires shape (B,C,T) instead of (B,T,C), which is confusing, so we combine the first two dims.
            # Remember: logits (B,T,vocab_size) matrix stores logit values for all vocabulary tokens at each position.
            # Cross-entropy converts logits to probabilities for each vocabulary token at every position.
            # It then selects the probability of the target token to compute the loss.
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, input, max_new_tokens):
        """
        Generates new tokens using past context.

        This function predicts next tokens one by one up to max_new_tokens.
        It crops the input context to the maximum block size before each step.

        Parameters:
            input (torch.Tensor): Tensor of shape (B, T) with current token indices.
            max_new_tokens (int): The number of new tokens to generate.

        Returns:
            torch.Tensor: Tensor of shape (B, T + max_new_tokens) with full sequence.
        """
        for _ in range(max_new_tokens):
            # limit context to latest block_size tokens
            input_clipped_to_context_window = input[:, -block_size:]

            # forward pass this temporary slice through the model
            logits, _ = self.forward(x=input_clipped_to_context_window, targets=None)  # logits is (B,T,vocab_size)

            # focus only on the last timestep; no dependence on past here
            # Reason: through self-attention trick, the last position/timestep already takes into account
            # all the past positions for every batch.
            logits = logits[:, -1, :]  # (B,vocab_size)

            # convert logits to probabilities
            # for every vocab token for each batch, we get one probability value
            probs = F.softmax(logits, dim=-1)  # (B,vocab_size)

            # sample next token from probability distribution for every batch
            next_token = torch.multinomial(probs, num_samples=1)  # (B,1)

            # append next_token to the original context
            input = torch.cat((input, next_token), dim=1)  # (B,T+1)
        return input


# -------------------------------single forward pass for testing-------------------------------
# model = BigramModel()
# m = model.to(device)
# # number of parameters in the model
# print(sum(p.numel() for p in m.parameters()) / 1e6, "M parameters")

# # single forward pass
# xb, yb = get_batch(split="train")
# logits, loss = m(x=xb, targets=yb)
# print(f"Loss = {loss}")

# -------------------------------train the model e2e-------------------------------

# call the model
model = BigramModel()
m = model.to(device)

# number of parameters in the model
print(sum(p.numel() for p in m.parameters()) / 1e6, "M parameters")


# train and loss computation
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = m(x=X, targets=Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out


optimizer = torch.optim.AdamW(m.parameters(), lr=lr)

for step in range(max_iters):
    # periodically evaluate the mean loss for multiple train and val batches
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"Step {step}: Train loss {losses['train']:.4f}, Val loss {losses['val']:.4f}")

    # sample batch data
    xb, yb = get_batch("train")

    logits, loss = m(x=xb, targets=yb)
    # set all gradients None
    optimizer.zero_grad(set_to_none=True)
    # run backward pass
    loss.backward()
    # make gradient update for each parameter
    optimizer.step()

    # # print loss every few iterations
    # if step % 1000 == 0:
    #     print(f"Iter: {step} | Loss: {loss.item()}")

# print(f"Final Loss: {loss.item()}")

# -------------------------------generate using the model -------------------------------

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(input=context, max_new_tokens=2000)[0].tolist()))


Using device: mps
0.209601 M parameters
Step 0: Train loss 4.6082, Val loss 4.6058
Step 100: Train loss 2.6389, Val loss 2.6381
Step 200: Train loss 2.5640, Val loss 2.5768
Step 300: Train loss 2.5350, Val loss 2.5603
Step 400: Train loss 2.5282, Val loss 2.5368
Step 500: Train loss 2.5209, Val loss 2.5397
Step 600: Train loss 2.5095, Val loss 2.5260
Step 700: Train loss 2.5128, Val loss 2.5198
Step 800: Train loss 2.5032, Val loss 2.5134
Step 900: Train loss 2.4988, Val loss 2.5066
Step 1000: Train loss 2.5034, Val loss 2.5178
Step 1100: Train loss 2.4934, Val loss 2.5103
Step 1200: Train loss 2.5018, Val loss 2.5142
Step 1300: Train loss 2.4877, Val loss 2.5071
Step 1400: Train loss 2.4890, Val loss 2.5122
Step 1500: Train loss 2.4974, Val loss 2.5199
Step 1600: Train loss 2.4905, Val loss 2.5108
Step 1700: Train loss 2.4864, Val loss 2.5074
Step 1800: Train loss 2.4824, Val loss 2.5082
Step 1900: Train loss 2.4816, Val loss 2.4980
Step 2000: Train loss 2.4786, Val loss 2.5053
Step 2

In [2]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(input=context, max_new_tokens=20000)[0].tolist()))


ET:
cestit d oneyoor is
DUSith S beaineindouemy ileloneele acht t y IXE:
KI d IThe HAle mean fr ssestandito heronf tha ake acou pe haitrren:
Wiflathy ly woolinchit sh I s or.
DUTomel thed theom, y tr fungt.
Hande, OMR:
NG ie tolaingovy ndor!
INS:
CHAlls!
Bulenditotorglayowob I ty ourgerok hantewe'thavits,

OF s, d I ge'll.
COFLomans k.
Th

Ingeray apers'd. hth,
AUCARSClal wirscathourea anke Wharr-we in mbenond o oray fr ichayoprichenghel.

So I mangn ting ng oss bel gn n, uis iet l!

Frthelens
ad pe ou qur pe I'thimamy YCHowizes.

tshd io ag r mn s t an, win baveare s f os ppomelllowil theserrio.

Me owharin,
ALAMI:
DY the wis, froniterow; wat ARESThowincey? d:
A:

Moon gly, m mphedwicr cus in a I I:
HENIn y; sendilenoucle, y t; it
CENGrenfolld co a t
M:
My HAnge'dein'sulou, I w owoard mer t inso y'de: airamango INRENowigakl be.
'd he ngake:
ARCIZALI ETods ond wainge Somboner binnd jo otovethe sho tot;
Hoffe opourean mends avelowser cutownolily dounct, mellind?
Andeasud be at lays ve,